[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tnecniv-Teikram/NSA-hyperspectral-course-2026/blob/main/exercises/ex_04_fire_disasters.ipynb)

*Running in Colab? Run the setup cell below first — it clones the course and installs what this notebook needs. Running locally? It does nothing.*

# 🔥 Exercises: Climate-induced disasters: fire mapping
### Space Academy 2026 · Hyperspectral course · Week 2 — Project scoping and use case definition (Practical)

Turn the burn indices of notebook 04 into the numbers an emergency agency needs — hectares, perimeters and severity — for the December 2024 Franklin fire in Malibu, test how robust they are, and (optionally) compare two Tanager dates to map the January 2025 Palisades fire with dNBR.

These exercises accompany the theme notebook [`04_climate_disasters_fire_flood.ipynb`](../../notebooks/04_climate_disasters_fire_flood.ipynb). Run that notebook first
(or at least read it) — here we reuse its scene and indices, loaded with the course helper module `hsi_utils`.

> Work through the exercises in order — they get harder (⭐ → ⭐⭐⭐). Each has a hint; worked solutions are in `exercises/solutions/`.

**What you will practise**
- Burned-area statistics from NBR and their threshold sensitivity
- From noisy pixels to a fire perimeter with morphology and connected components
- Reading burn-scar spectra: what fire changes and what NBR captures
- Co-registering two scenes on the same grid and computing dNBR
- Why severity is defined on change (dNBR), not on single-date NBR

**Data**
- `20241215_185916_33_4001` — Malibu, California (15 Dec 2024), after the Franklin fire (9–17 Dec 2024)
- *Optional:* `20250123_185518_92_4001` — Santa Monica Mountains (23 Jan 2025), after the Palisades fire

In [ ]:
# --- Setup: works both in Google Colab and in a local environment --------------------
# In Colab this clones the course repository, installs the geospatial packages that Colab
# does not ship, and moves into the course folder so the notebook finds hsi_utils and data/.
# Locally it does nothing at all — your own environment is already active.
import os
import pathlib
import subprocess
import sys

REPO = "Tnecniv-Teikram/NSA-hyperspectral-course-2026"          # change with: python tools/set_repo_url.py <owner>/<repo>

if "google.colab" in sys.modules:
    course = pathlib.Path("/content/hyperspectral-course")
    if not course.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-q",
                        f"https://github.com/{REPO}.git", str(course)], check=True)
    os.chdir(course)
    sys.path.insert(0, str(course))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "h5py", "rasterio", "rioxarray", "xarray", "geopandas"], check=True)
    print("Colab ready — course at", course)
else:
    print("Local environment:", sys.executable.split("/")[-3])


## Setup

In [ ]:
# hsi_utils.py lives in the course folder (.. from here)
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

# On Google Colab: upload hsi_utils.py next to this notebook, change the line above to
# sys.path.insert(0, ".") and uncomment:
# %pip install -q h5py numpy scipy matplotlib scikit-learn pandas rasterio requests

import numpy as np
import matplotlib.pyplot as plt
import hsi_utils as hu

plt.rcParams["figure.dpi"] = 90

Load the scene and the bands used in the theme notebook. `hu.get_scene()` finds the local HDF5 or downloads it (~1 GB) from Planet's STAC catalogue.

In [ ]:
from scipy import ndimage

SCENE_ID = "20241215_185916_33_4001"   # Malibu, California — 15 Dec 2024, days after the Franklin fire
scene = hu.get_scene(SCENE_ID)
print(scene.summary())

valid = scene.valid_mask()
R_green, R_red, R_nir, R_swir2 = (scene.read_band(w) for w in (560, 665, 860, 2200))   # as in notebook 04
NBR = hu.nd(R_nir, R_swir2)
NDWI = hu.nd(R_green, R_nir)
sea = valid & (NDWI > 0.1)
land = valid & ~sea
PIXEL_HA = 0.09

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].imshow(scene.rgb()); axes[0].set_title("True colour")
im = axes[1].imshow(np.where(land, NBR, np.nan), cmap="RdYlGn", vmin=-0.5, vmax=0.7)
axes[1].set_title("NBR over land (sea masked)"); plt.colorbar(im, ax=axes[1], fraction=0.035)
for a in axes: a.axis("off")
plt.tight_layout(); plt.show()

## Exercise 1 — Burned area in hectares  ⭐

1. Count the land pixels with NBR < −0.1 and convert the count to hectares.
2. Reproduce the five severity classes of notebook 04 (NBR ≥ 0.1 unburned; −0.1…0.1 low; −0.27…−0.1
   moderate-low; −0.44…−0.27 moderate-high; < −0.44 high) and make a table of the area per class, over land only.

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

`np.digitize(NBR, bins)` with `bins = [-0.44, -0.27, -0.1, 0.1]` assigns each pixel to one of five intervals
(0 = lowest NBR = high severity). Then use `np.bincount` on the land pixels.

</details>

## Exercise 2 — How sensitive is the burned area to the threshold?  ⭐⭐

Plot the burned area (ha) as a function of the NBR threshold, from −0.5 to +0.2. Mark the value −0.1 used above.
Where is the curve flat (the estimate is robust) and where is it steep? What does a steep section mean
physically?

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

`ts = np.linspace(-0.5, 0.2, 71)` and `[(land & (NBR < t)).sum() * PIXEL_HA for t in ts]`. A histogram of NBR
over land helps to interpret the curve (the area curve is its cumulative sum).

</details>

## Exercise 3 — From pixels to a fire perimeter  ⭐⭐

Agencies report fires as **polygons** (perimeters), not as scattered pixels.

1. Clean the `NBR < −0.1` mask with a morphological **closing** (2 iterations) to fill small unburned gaps.
2. Label the connected regions and keep the largest one — the Franklin fire scar.
3. Report its area and compare it with the ~1,630 ha (4,000 acres) reported by CAL FIRE. Draw its outline on the
   true-colour image.
4. Why might your area be smaller or larger than the official one?

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

`ndimage.binary_closing(mask, iterations=2)`, `labels, n = ndimage.label(mask)`,
`sizes = ndimage.sum(mask, labels, range(1, n + 1))`. Draw the outline with `ax.contour(region, [0.5], colors="red")`.

</details>

## Exercise 4 — Burn scar vs chaparral: the spectra behind NBR  ⭐⭐

Plot the mean spectrum (± one standard deviation) of burned land (NBR < −0.15) and of unburned chaparral /
shrubland (NBR > 0.3). Mark the NBR bands (860 and 2200 nm). Which spectral changes caused by the fire does NBR
capture, and which does it ignore?

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

Sample up to ~500 pixels per class with `np.random.default_rng(0).choice`, read the bounding window with
`scene.read_cube` and index the sampled pixels. `ax.fill_between(wl, mean - std, mean + std, alpha=0.3)`.

</details>

## Exercise 5 — Before and after: dNBR with a second Tanager scene (optional)  ⭐⭐⭐

Tanager imaged the same coast again on **23 January 2025** (`20250123_185518_92_4001`), after the Palisades
fire of 7 January 2025. That scene is not part of the standard course download (~1.3 GB). Set
`DOWNLOAD_POST = True` to fetch it, or skip this exercise.

1. Both scenes are on the same UTM grid (EPSG:32611, 30 m). Use their geotransforms to compute the row/column
   offset between them and cut the overlapping area out of both NBR maps.
2. Check the co-registration: do the coastlines (NDWI > 0.1) line up?
3. Compute dNBR = NBR(15 Dec) − NBR(23 Jan) over land in both dates and classify it with the USGS thresholds
   (Key & Benson, 2006): < 0.1 unburned, 0.1–0.27 low, 0.27–0.44 moderate-low, 0.44–0.66 moderate-high,
   > 0.66 high.
4. What is dNBR inside the Franklin scar, and why? Where is the new burn?

In [ ]:
# your code here

<details><summary>💡 Hint</summary>

`scene.transform().c` is the x (easting) of the upper-left corner and `.f` its y (northing). A pixel at pre
(row, col) sits at post (row + (y_post − y_pre)/30, col + (x_pre − x_post)/30) — rows increase southward while y
increases northward. Clip the window so it stays inside both grids.

</details>

## Exercise 6 — Reflection — NBR, dNBR and what else is dark in the SWIR  ⭐

Answer in a short markdown cell:

1. Why are the burn-severity thresholds of Key & Benson (2006) defined for **dNBR** and not for single-date NBR?
2. Name three non-burned surfaces that can have NBR < 0 in a single image, and one way to exclude each of them.
3. How soon after a fire should the post-fire image be taken, and why does it matter?

In [ ]:
# optional scratch space — answer in a markdown cell

<details><summary>💡 Hint</summary>

Think about what is different between two pixels with the same NBR but different pre-fire vegetation, and about
vegetation regrowth, ash being blown away and winter rains.

</details>

## Key takeaways

- Burned-area numbers depend on the threshold — look for the gap in the NBR histogram.
- Morphology and connected components turn pixel masks into reportable perimeters.
- Fire darkens the NIR and brightens the SWIR; hyperspectral data also sees the lost red edge and leaf water.
- Scenes on the same grid can be differenced after an integer offset; always check registration first.
- Severity thresholds apply to dNBR; single-date NBR confuses burns with rock, roads and water.

---

> 📖 Data: Tanager — CC-BY-4.0 © Planet Labs PBC